# Lane Seven — Demand Forecasting
# **v7.6 — Conservative Global Bias Control** 🛡️

> **v7.5 per-StyleCode calibration worsened performance.**
> v7.6 applies only a global HorizonMonths-level correction with a no-regression safety gate.

## What v7.6 adds over v7.4

| Layer | v7.4 | v7.6 |
|-------|------|------|
| Upstream model | StyleCode XGB/LGBM | **same** |
| Calibration | Optional, often 1.0 | **Global factor per HorizonMonths** |
| Factor bounds | [0.80, 1.20] | **[0.95, 1.05] — intentionally conservative** |
| No-regression gate | No | **Yes — rejects calibration if WMAPE worsens >0.25pp** |
| Allocation | recency_only | **same (v7.2 champion)** |

## Key design decisions

- **Global only**: one factor per HorizonMonths (not per StyleCode)
- **Conservative bounds**: [0.95, 1.05] prevents over-correction
- **No-regression gate**: if calibration worsens WMAPE, factor = 1.0 (rejected)
- **Evidence priority**: v7.5 backtest predictions → v7.4 holdout (diagnostic) → 1.0
- **Leakage safe**: pre-2026 backtest data used; holdout data labeled 'diagnostic'

## Required inputs

```
data/gold_fact_monthly_demand_v2.csv
data/dim_date.csv
data/dim_product.csv
outputs/v7_4_best_models.csv  ← or any prior v7.x best models
outputs/v7_5_stylecode_backtest_predictions.csv  ← preferred evidence (optional)
outputs/v7_4_holdout_predictions.csv  ← diagnostic fallback (optional)
```

## Output files

```
outputs/v7_6_global_calibration_table.csv
outputs/v7_6_stylecode_forecasts_raw.csv
outputs/v7_6_stylecode_forecasts_calibrated.csv
outputs/v7_6_production_sku_forecasts.csv    ← MAIN DELIVERABLE
outputs/v7_6_holdout_evaluation.csv
outputs/v7_6_error_decomposition.csv
outputs/v7_6_bias_analysis.csv
outputs/v7_6_vs_prior_versions.csv
outputs/v7_6_validation_report.csv
```

## Section 0 — Environment setup

In [1]:
import subprocess, sys
packages = ['pandas','numpy','scikit-learn','xgboost','lightgbm','statsmodels','openpyxl']
subprocess.check_call([sys.executable,'-m','pip','install','--quiet']+packages)
print('All packages installed.')

All packages installed.


In [2]:
import sys, logging
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

NOTEBOOK_DIR = Path('.').resolve()
DATA_DIR     = NOTEBOOK_DIR / 'data'
OUTPUT_DIR   = NOTEBOOK_DIR / 'outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

logging.basicConfig(
    level=logging.INFO, format='%(asctime)s  %(levelname)-8s  %(message)s',
    datefmt='%H:%M:%S', handlers=[logging.StreamHandler(sys.stdout)], force=True,
)
for _noisy in ['lightgbm','xgboost','prophet','cmdstanpy']:
    logging.getLogger(_noisy).setLevel(logging.ERROR)

GOLD_OPTIONS = ['gold_fact_monthly_demand_v2.csv','gold_fact_monthly_demand.csv']
gold_present = [f for f in GOLD_OPTIONS if (DATA_DIR/f).exists()]
print(f'DATA_DIR   : {DATA_DIR}')
print(f'OUTPUT_DIR : {OUTPUT_DIR}')
print(f'  {chr(10003) if gold_present else chr(10007)}  {gold_present[0] if gold_present else "MISSING"}')
print(f'  {chr(10003) if (DATA_DIR/"dim_date.csv").exists() else chr(10007)}  dim_date.csv')
print(f'  {chr(10003) if (DATA_DIR/"dim_product.csv").exists() else chr(10007)}  dim_product.csv')
if not gold_present:
    raise FileNotFoundError('Gold demand file missing.')
print('\nSetup complete.')

DATA_DIR   : C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.6\data
OUTPUT_DIR : C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.6\outputs
  ✓  gold_fact_monthly_demand_v2.csv
  ✓  dim_date.csv
  ✓  dim_product.csv

Setup complete.


## Section 1 — Configuration

In [3]:
from lane7_forecast.forecast_adjustments import get_config

PHASE = 1

# -----------------------------
# Dynamic holdout detection
# -----------------------------
_dim_date_check = pd.read_csv(
    DATA_DIR / "dim_date.csv",
    parse_dates=["MonthStart"]
)

_dim_date_check["MonthStart"] = (
    _dim_date_check["MonthStart"]
    .dt.to_period("M")
    .dt.to_timestamp()
)

# Check common split column names safely
_candidate_split_cols = [
    "Split",
    "TrainHoldout",
    "DataSplit",
    "DatasetSplit",
    "Set",
    "IsTrain"
]

_holdout_mask = None
_holdout_source_col = None

for _col in _candidate_split_cols:
    if _col not in _dim_date_check.columns:
        continue

    if _col == "IsTrain":
        _vals = _dim_date_check[_col]

        if _vals.dtype == bool:
            _mask = (_vals == False)
        else:
            _mask = (
                _vals.astype(str)
                .str.strip()
                .str.lower()
                .isin(["0", "false", "holdout"])
            )

    else:
        _mask = (
            _dim_date_check[_col]
            .astype(str)
            .str.strip()
            .str.upper()
            .eq("HOLDOUT")
        )

    if _mask.any():
        _holdout_mask = _mask
        _holdout_source_col = _col
        break

if _holdout_mask is None:
    raise ValueError(
        "Could not detect holdout months from dim_date.csv.\n"
        "Expected one of:\n"
        "- Split / TrainHoldout / DataSplit / DatasetSplit / Set = HOLDOUT\n"
        "- OR IsTrain = False/0"
    )

HOLDOUT_MONTHS = sorted(
    _dim_date_check.loc[_holdout_mask, "MonthStart"].unique()
)

HOLDOUT_MONTHS = [pd.Timestamp(m) for m in HOLDOUT_MONTHS]

if len(HOLDOUT_MONTHS) == 0:
    raise ValueError("No holdout months detected.")

HOLDOUT_START = HOLDOUT_MONTHS[0].strftime("%Y-%m-%d")
N_HOLDOUT_MONTHS = len(HOLDOUT_MONTHS)

# Needed for v7.6 leakage guard
BACKTEST_END = HOLDOUT_MONTHS[0] - pd.DateOffset(months=1)
BACKTEST_END = pd.Timestamp(
    BACKTEST_END.year,
    BACKTEST_END.month,
    1
)

print(f"Detected holdout source column: {_holdout_source_col}")
print("Detected holdout months:")
for m in HOLDOUT_MONTHS:
    print(" ", m.strftime("%Y-%m"))

print(f"Backtest end: {BACKTEST_END.strftime('%Y-%m')}")

# -----------------------------
# Existing forecast adjustment config
# -----------------------------
ADJ_CONFIG = get_config(
    shrink_threshold=1.3,
    shrink_factor=0.75,
    recursive_alpha=0.6,
    blend_threshold=0.20,
    blend_weight=0.7,
    intermittent_cap_multiplier=2.0,
)

# -----------------------------
# Allocation config
# -----------------------------
ALLOC_PARAMS = dict(
    lookback_months=12,
    min_lookback_months=6,
    w_recent=3,
    w_mid=2,
    w_old=1
)

# -----------------------------
# v7.6 calibration config
# -----------------------------
V76_CALIB_CONFIG = dict(
    min_factor=0.95,
    max_factor=1.05,
    max_allowed_wmape_regression=0.25
)

print("\n=== v7.6 Calibration Config ===")
for k, v in V76_CALIB_CONFIG.items():
    print(f"{k:35}: {v}")

print()
print("Factor bounds are intentionally narrow to prevent over-correction.")
print("No-regression gate will reject calibration if WMAPE worsens > 0.25pp.")

Detected holdout source column: Split
Detected holdout months:
  2026-01
  2026-02
  2026-03
  2026-04
Backtest end: 2025-12

=== v7.6 Calibration Config ===
min_factor                         : 0.95
max_factor                         : 1.05
max_allowed_wmape_regression       : 0.25

Factor bounds are intentionally narrow to prevent over-correction.
No-regression gate will reject calibration if WMAPE worsens > 0.25pp.


## Section 2 — Load Data and Build StyleCode Panel

In [4]:
from lane7_forecast.pipeline import run_v7_prep, run_cv

scode_prep = run_v7_prep(
    data_dir=DATA_DIR, phase=PHASE,
    lookback_months=ALLOC_PARAMS['lookback_months'],
    min_lookback_months=ALLOC_PARAMS['min_lookback_months'],
)

tables      = scode_prep['tables']
gold_df     = tables['demand']
dim_prod_df = scode_prep['dim_product']
dim_date_df = tables['dim_date']

scode_segments = scode_prep['segments']
scode_segments.to_csv(OUTPUT_DIR/'v7_6_stylecode_segments.csv', index=False)

print(f'StyleCode panel: {scode_prep["panel_seg"]["SKU"].nunique():,} StyleCodes')
for seg, n in scode_segments['Segment'].value_counts().items():
    print(f'  {seg:<15} {n:5d}')

15:41:15  INFO      === v7 Step 1: StyleCode-level data prep (phase=1) ===
15:41:15  INFO      Loaded gold_fact_monthly_demand: 119498 rows, 4 cols
15:41:15  INFO      Loaded dim_date: 116 rows, 6 cols
15:41:15  INFO      Loaded dim_product: 3806 rows, 15 cols
15:41:15  WARNING   Optional file not found, skipping: C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.6\data\dim_customer.csv
15:41:15  INFO      Training period end (phase 1): 2025-12
15:41:15  INFO      [v7] STANDALONE (StyleCode level): 468 SKUs, 538 rows, 19,551 units (0.03% of total)
15:41:15  INFO      [v7] StyleCode demand table: 2336 rows, 61 unique StyleCodes, 2017-05 → 2026-04
15:41:15  INFO      Training period end (phase 1): 2025-12
15:41:15  INFO      Rows after date filter (<= 2025-12): 2190
15:41:16  WARNING   Attribute columns missing from demand and no dim_product provided: ['Category', 'StyleCode', 'ColorCode', 'SizeCode', 'StyleColor']. Filling with 'UNKNOWN'.
15:41:16  INFO   

In [5]:
USE_EXISTING_CV = False
_best_path = OUTPUT_DIR / 'v7_6_best_models.csv'
_v74_best  = OUTPUT_DIR / 'v7_4_best_models.csv'
_v75_best  = OUTPUT_DIR / 'v7_5_best_models.csv'

v76_bm = {}

if USE_EXISTING_CV and _best_path.exists():
    print('Loading cached v7.6 CV results...')
    best_models_v76 = pd.read_csv(_best_path)
    for h in [1,3]:
        v76_bm[h] = best_models_v76[best_models_v76['HorizonMonths']==h].copy()
elif _v74_best.exists():
    print(f'Reusing v7.4 best models from {_v74_best.name}')
    best_models_v76 = pd.read_csv(_v74_best)
    for h in [1,3]:
        v76_bm[h] = best_models_v76[best_models_v76['HorizonMonths']==h].copy()
else:
    for h, cfg in {1:dict(n_folds=6,step_months=3,min_train_months=24),
                   3:dict(n_folds=5,step_months=3,min_train_months=24)}.items():
        print(f'--- StyleCode CV H={h} ---')
        cv_df, best_df = run_cv(scode_prep, horizon_months=h, **cfg)
        v76_bm[h] = best_df

best_models_v76 = pd.concat([df for df in v76_bm.values() if not df.empty], ignore_index=True)
best_models_v76.to_csv(OUTPUT_DIR/'v7_6_best_models.csv', index=False)

print('\n=== v7.6 Best Models ===')
print(best_models_v76[['Segment','HorizonMonths','BestModel','mean_WMAPE']].round(2).to_string(index=False))

--- StyleCode CV H=1 ---
15:41:37  INFO      === Step 2: CV for horizon=1 months ===
15:41:37  INFO      Horizon 1-month: dropped 156 rows with null required lags (['lag_1', 'lag_2', 'lag_3', 'rolling_mean_3'])
15:41:37  INFO      Features created for horizon=1 — panel shape: (3694, 34)
15:41:37  INFO      CV: segment=REGULAR, horizon=1, models=['XGBoost', 'LightGBM', 'SeasonalNaive'], folds=6
15:41:43  INFO      Trained XGBoost on 1331 rows, 16 features
15:41:51  INFO      Trained LightGBM on 1331 rows, 16 features
15:42:00  INFO      Trained XGBoost on 1407 rows, 16 features
15:42:16  INFO      Trained LightGBM on 1407 rows, 16 features
15:42:21  INFO      Trained XGBoost on 1485 rows, 16 features
15:42:26  INFO      Trained LightGBM on 1485 rows, 16 features
15:42:30  INFO      Trained XGBoost on 1565 rows, 16 features
15:42:35  INFO      Trained LightGBM on 1565 rows, 16 features
15:42:38  INFO      Trained XGBoost on 1646 rows, 16 features
15:42:42  INFO      Trained LightGBM on 1

## Section 3 — Load Calibration Evidence

In [6]:
# Load evidence in priority order:
#   1. v7.5 backtest predictions (pre-2026, preferred)
#   2. v7.4 holdout predictions (diagnostic fallback)

_bt_path   = OUTPUT_DIR / 'v7_5_stylecode_backtest_predictions.csv'
_hold_path = OUTPUT_DIR / 'v7_4_holdout_predictions.csv'

backtest_df = None
holdout_df  = None

if _bt_path.exists():
    backtest_df = pd.read_csv(_bt_path)
    print(f'Loaded v7.5 backtest: {len(backtest_df):,} rows')
    print(f'  HorizonMonths: {sorted(backtest_df["HorizonMonths"].unique())}')
    print(f'  ForecastMonths: {sorted(backtest_df["ForecastMonth"].unique())[:5]} ...')
    print(f'  StyleCodes: {backtest_df["StyleCodeDesc"].nunique():,}')
else:
    print(f'{_bt_path.name} not found.')
    print('Will try diagnostic holdout fallback.')

if _hold_path.exists():
    holdout_df = pd.read_csv(_hold_path, parse_dates=['MonthStart'])
    holdout_df['MonthStart'] = holdout_df['MonthStart'].dt.to_period('M').dt.to_timestamp()
    # Rename columns to standard names
    if 'PredictedUnits' not in holdout_df.columns and 'ForecastUnits' in holdout_df.columns:
        holdout_df = holdout_df.rename(columns={'ForecastUnits':'PredictedUnits'})
    print(f'Loaded v7.4 holdout: {len(holdout_df):,} rows (diagnostic fallback)')
else:
    print(f'{_hold_path.name} not found — will use factor = 1.0')

if backtest_df is None and holdout_df is None:
    print('\nNo calibration evidence available. Will use factor = 1.0 for all horizons.')

v7_5_stylecode_backtest_predictions.csv not found.
Will try diagnostic holdout fallback.
v7_4_holdout_predictions.csv not found — will use factor = 1.0

No calibration evidence available. Will use factor = 1.0 for all horizons.


## Section 4 — Generate Raw StyleCode Forecasts

In [7]:
print("DATA_DIR:", DATA_DIR)
print("HOLDOUT_MONTHS:", [m.strftime("%Y-%m") for m in HOLDOUT_MONTHS])
print("HOLDOUT_START:", HOLDOUT_START)
print("N_HOLDOUT_MONTHS:", N_HOLDOUT_MONTHS)

_dim_check = pd.read_csv(DATA_DIR / "dim_date.csv", parse_dates=["MonthStart"])
print(_dim_check.tail(12))

DATA_DIR: C:\Users\TimothySpencerTan\Downloads\Lane 7 Predictive Model v7\PredictiveModel7.6\data
HOLDOUT_MONTHS: ['2026-01', '2026-02', '2026-03', '2026-04']
HOLDOUT_START: 2026-01-01
N_HOLDOUT_MONTHS: 4
    MonthStart  Year  Month  Quarter     Split  IsTrain
104 2026-01-01  2026      1        1   HOLDOUT        0
105 2026-02-01  2026      2        1   HOLDOUT        0
106 2026-03-01  2026      3        1   HOLDOUT        0
107 2026-04-01  2026      4        2   HOLDOUT        0
108 2026-05-01  2026      5        2  FORECAST        0
109 2026-06-01  2026      6        2  FORECAST        0
110 2026-07-01  2026      7        3  FORECAST        0
111 2026-08-01  2026      8        3  FORECAST        0
112 2026-09-01  2026      9        3  FORECAST        0
113 2026-10-01  2026     10        4  FORECAST        0
114 2026-11-01  2026     11        4  FORECAST        0
115 2026-12-01  2026     12        4  FORECAST        0


In [ ]:
from lane7_forecast.pipeline import run_forecasts
from lane7_forecast.data_prep import _resolve_training_end, build_panel
from lane7_forecast.segmentation import segment_skus, attach_segment

train_end       = _resolve_training_end(dim_date_df, phase=1)
standalone_skus = scode_prep.get('stylecode_standalone_skus', [])

v76_raw_fc = {}
v76_sa_fc  = {}

for horizon in [3, 1]:
    bm_h = best_models_v76[best_models_v76['HorizonMonths']==horizon]
    if bm_h.empty:
        print(f'H={horizon}: no best model — skipping.')
        continue
    n_pred = N_HOLDOUT_MONTHS

    raw_fc = run_forecasts(
        prep=scode_prep, best_models_df=bm_h, horizon_months=horizon,
        forecast_start=HOLDOUT_START, n_forecast_months=n_pred,
        phase=PHASE, model_version=f'v7.6-raw-H{horizon}',
        output_path=None, append=False, adjustment_config=ADJ_CONFIG,
    )
    v76_raw_fc[horizon] = raw_fc
    raw_fc.to_csv(OUTPUT_DIR/f'v7_6_stylecode_forecasts_raw_H{horizon}.csv', index=False)
    print(f'H={horizon} raw: {len(raw_fc):,} rows, {raw_fc["Key"].nunique():,} StyleCodes')

    if standalone_skus:
        sa_gold = gold_df[gold_df['SKU'].isin(standalone_skus)].copy()
        if not sa_gold.empty:
            sa_p = build_panel(sa_gold, dim_date_df, phase=PHASE, dim_product_df=dim_prod_df)
            sa_s = segment_skus(sa_p)
            sa_prep = {'tables':tables,'panel':sa_p,'segments':sa_s,
                       'panel_seg':attach_segment(sa_p,sa_s)}
            v76_sa_fc[horizon] = run_forecasts(
                prep=sa_prep, best_models_df=bm_h, horizon_months=horizon,
                forecast_start=HOLDOUT_START, n_forecast_months=n_pred,
                phase=PHASE, model_version=f'v7.6-standalone-H{horizon}',
                output_path=None, append=False, adjustment_config=ADJ_CONFIG,
            )

raw_combined = pd.concat([df for df in v76_raw_fc.values() if not df.empty], ignore_index=True)
raw_combined.to_csv(OUTPUT_DIR/'v7_6_stylecode_forecasts_raw.csv', index=False)
print(f"H={horizon} | n_pred={n_pred} | HOLDOUT_START={HOLDOUT_START}")
print(f'\nSaved v7_6_stylecode_forecasts_raw.csv ({len(raw_combined):,} rows)')

15:43:19  INFO      Training period end (phase 1): 2025-12
15:43:19  INFO      === Step 3: Forecast generation (horizon=3, phase=1) ===
15:43:20  INFO      Horizon 3-month: dropped 590 rows with null required lags (['lag_1', 'lag_3', 'lag_6', 'lag_12', 'rolling_mean_6'])
15:43:20  INFO      Features created for horizon=3 — panel shape: (3260, 34)
15:43:24  INFO      Trained XGBoost on 1485 rows, 19 features
15:43:24  INFO      Retrained XGBoost for segment=REGULAR on 1485 rows
15:43:24  INFO      Generating 3-month forecasts for 52 SKUs, 4 months from 2026-01 (adjustments=enabled)
15:43:29  INFO      Forecast complete: 208 rows, horizon=3
H=3 raw: 208 rows, 52 StyleCodes
15:43:29  INFO      Training period end (phase 1): 2025-12
15:43:29  INFO      Rows after date filter (<= 2025-12): 537
15:43:32  INFO      dim_product joined onto panel. Unknown counts per attr: {'StyleCode': 26877}
15:43:32  INFO      Panel built: 26961 rows, 468 unique SKUs, date range 2017-12 to 2025-12
15:43:33  I

## Section 5 — Build Global Calibration Table

**No-regression gate:** if calibration would worsen WMAPE > 0.25pp at any horizon,
that horizon's calibration is rejected and factor = 1.0 is used instead.

In [9]:
from lane7_forecast.global_bias_control_v76 import (
    build_global_calibration_table, validate_global_calibration_table,
)

# Raw forecasts for the no-regression gate validation set
# Use H=3 raw as the representative forecast for gate computation
_gate_fc = v76_raw_fc.get(3, pd.DataFrame())

actuals_df_full = pd.read_csv(
    next(DATA_DIR/f for f in ['gold_fact_monthly_demand_v2.csv','gold_fact_monthly_demand.csv']
         if (DATA_DIR/f).exists()),
    parse_dates=['MonthStart']
)
actuals_df_full['MonthStart'] = actuals_df_full['MonthStart'].dt.to_period('M').dt.to_timestamp()
actuals_df_full['SKU']        = actuals_df_full['SKU'].astype(str).str.strip()

calib_table = build_global_calibration_table(
    backtest_predictions_df=backtest_df,
    holdout_predictions_df=holdout_df,
    raw_scode_forecasts_df=_gate_fc,
    actuals_df=actuals_df_full,
    dim_product_df=dim_prod_df,
    holdout_months=HOLDOUT_MONTHS,
    horizon_months_list=[1, 3],
    **V76_CALIB_CONFIG,
)
calib_table.to_csv(OUTPUT_DIR/'v7_6_global_calibration_table.csv', index=False)

val_calib = validate_global_calibration_table(
    calibration_df=calib_table,
    min_factor=V76_CALIB_CONFIG['min_factor'],
    max_factor=V76_CALIB_CONFIG['max_factor'],
)

print('=== v7.6 Global Calibration Table ===')
print(calib_table[[
    'HorizonMonths','raw_bias_ratio','proposed_factor','final_factor',
    'raw_wmape','calibrated_wmape','wmape_delta',
    'calibration_applied','rejection_reason','evidence_source','n_observations'
]].to_string(index=False))
print()
print(f'Validation: n_applied={val_calib["n_applied"]}  '
      f'in_range={val_calib["factors_in_range"]}  '
      f'gate_passed={val_calib["no_regression_gate_passed"]}')
if val_calib['warnings']:
    print(f'Warnings: {val_calib["warnings"]}')

15:44:01  INFO      [v7.6] Global calibration table: 2 horizons, 0 calibration(s) applied
=== v7.6 Global Calibration Table ===
 HorizonMonths raw_bias_ratio  proposed_factor  final_factor  raw_wmape  calibrated_wmape  wmape_delta  calibration_applied           rejection_reason evidence_source  n_observations
             1           None              1.0           1.0        NaN               NaN          NaN                False insufficient_safe_evidence            none               0
             3           None              1.0           1.0    30.2939           30.2939          0.0                False insufficient_safe_evidence            none               0

Validation: n_applied=0  in_range=True  gate_passed=True


## Section 6 — Apply Calibration

In [10]:
from lane7_forecast.global_bias_control_v76 import apply_global_calibration

v76_cal_fc = {}

for horizon in [3, 1]:
    if horizon not in v76_raw_fc:
        continue

    calib_h = calib_table[calib_table['HorizonMonths']==horizon]
    cal_fc  = apply_global_calibration(
        scode_forecasts_df=v76_raw_fc[horizon],
        calibration_df=calib_h,
    )
    v76_cal_fc[horizon] = cal_fc

    n_adj = int(cal_fc['CalibrationApplied'].sum()) if 'CalibrationApplied' in cal_fc.columns else 0
    factor_row = calib_h['final_factor'].values[0] if not calib_h.empty else 1.0
    applied_flag = calib_h['calibration_applied'].values[0] if not calib_h.empty else False
    status = 'ACCEPTED' if applied_flag else 'REJECTED (factor=1.0)'
    print(f'H={horizon}: factor={factor_row:.4f}  status={status}  rows_adjusted={n_adj}')

cal_combined = pd.concat([df for df in v76_cal_fc.values() if not df.empty], ignore_index=True)
cal_combined.to_csv(OUTPUT_DIR/'v7_6_stylecode_forecasts_calibrated.csv', index=False)
print(f'\nSaved v7_6_stylecode_forecasts_calibrated.csv ({len(cal_combined):,} rows)')

15:44:02  INFO      [v7.6] Applied global calibration: 0 / 208 rows adjusted
H=3: factor=1.0000  status=REJECTED (factor=1.0)  rows_adjusted=0
15:44:02  INFO      [v7.6] Applied global calibration: 0 / 208 rows adjusted
H=1: factor=1.0000  status=REJECTED (factor=1.0)  rows_adjusted=0

Saved v7_6_stylecode_forecasts_calibrated.csv (416 rows)


## Section 7 — Production Allocation: v7.2 recency_only

In [11]:
from lane7_forecast.allocation_v72 import ALLOCATION_VARIANTS, run_allocation_variant

v76_scol_fc = {}
v76_sku_fc  = {}

for horizon in [3, 1]:
    if horizon not in v76_cal_fc:
        continue

    sa_fc = v76_sa_fc.get(horizon, None)
    alloc = run_allocation_variant(
        variant_name='recency_only',
        variant_cfg=ALLOCATION_VARIANTS['recency_only'],
        scode_forecasts_df=v76_cal_fc[horizon],
        gold_df=gold_df,
        dim_product_df=dim_prod_df,
        train_end=train_end,
        standalone_fc_df=sa_fc,
        **ALLOC_PARAMS,
        smooth_alpha=0.0,
        cap_rel_increase=999.0,
    )
    v76_scol_fc[horizon] = alloc['stylecolor_forecasts']
    v76_sku_fc[horizon]  = alloc['sku_forecasts']

    val = alloc['validation']
    print(f'H={horizon}: sc→scol={val["sc_to_scol_sum_ok"]}  '
          f'scol→sku={val["scol_to_sku_sum_ok"]}  '
          f'no_neg={val["no_negatives"]}  '
          f'SKUs={alloc["sku_forecasts"]["Key"].nunique():,}')

15:44:02  INFO      [v7.2] Running variant 'recency_only': recency=True  smoothing=False  caps=False
15:44:28  INFO      [v7.2] Variant 'recency_only' validation: sc→scol=True (diff=0.0009)  scol→sku=True (diff=0.0003)  no_neg=True
H=3: sc→scol=True  scol→sku=True  no_neg=True  SKUs=3,882
15:44:28  INFO      [v7.2] Running variant 'recency_only': recency=True  smoothing=False  caps=False
15:44:58  INFO      [v7.2] Variant 'recency_only' validation: sc→scol=True (diff=0.0004)  scol→sku=True (diff=0.0003)  no_neg=True
H=1: sc→scol=True  scol→sku=True  no_neg=True  SKUs=3,882


## Section 8 — Build Production SKU Table

In [12]:
from lane7_forecast.production_outputs_v76 import build_v76_production_sku_table

all_sku = pd.concat([df for df in v76_sku_fc.values() if not df.empty], ignore_index=True)

prod_sku = build_v76_production_sku_table(
    sku_fc_df=all_sku,
    dim_product_df=dim_prod_df,
    calibration_df=calib_table,
    allocation_method='recency_only_v7.2',
)
prod_sku.to_csv(OUTPUT_DIR/'v7_6_production_sku_forecasts.csv', index=False)

print(f'Production SKU table: {len(prod_sku):,} rows, {prod_sku["SKU"].nunique():,} SKUs')
print(f'Columns: {list(prod_sku.columns)}')
print()
print('Sample (H=3):')
print(prod_sku[prod_sku['HorizonMonths']==3].head(5)[[
    'MonthStart','HorizonMonths','SKU','StyleCodeDesc',
    'ForecastUnits','CalibrationFactor','CalibrationApplied','CalibrationScope'
]].to_string(index=False))

15:45:11  INFO      [v7.6] Production SKU table: 31056 rows, 3882 SKUs
Production SKU table: 31,056 rows, 3,882 SKUs
Columns: ['MonthStart', 'HorizonMonths', 'SKU', 'StyleCodeDesc', 'StyleColorDesc', 'SizeDesc', 'ForecastUnits', 'Lower', 'Upper', 'ModelName', 'ModelVersion', 'AllocationMethod', 'CalibrationFactor', 'CalibrationApplied', 'CalibrationScope']

Sample (H=3):
MonthStart  HorizonMonths             SKU StyleCodeDesc  ForecastUnits  CalibrationFactor  CalibrationApplied CalibrationScope
2026-01-01              3     002 Vintage           NaN            0.0                1.0               False           global
2026-01-01              3 002 Vintage Tee           NaN            0.0                1.0               False           global
2026-01-01              3    004 V-Hoodie           NaN            0.0                1.0               False           global
2026-01-01              3    12000 Crop H           NaN            0.0                1.0               False         

## Section 9 — Validation Report

In [13]:
from lane7_forecast.production_outputs_v76 import build_v76_validation_report

if 3 in v76_cal_fc and 3 in v76_scol_fc and 3 in v76_sku_fc:
    val_report = build_v76_validation_report(
        scode_fc=v76_cal_fc[3],
        scol_fc=v76_scol_fc[3],
        sku_fc=v76_sku_fc[3],
        calibration_df=calib_table,
        min_factor=V76_CALIB_CONFIG['min_factor'],
        max_factor=V76_CALIB_CONFIG['max_factor'],
    )
    val_report.to_csv(OUTPUT_DIR/'v7_6_validation_report.csv', index=False)

    print('=== v7.6 Validation Report ===')
    print(val_report.to_string(index=False))
    n_fail = (~val_report['passed']).sum()
    print()
    print('✅ ALL CHECKS PASSED' if n_fail == 0 else f'❌ {n_fail} check(s) FAILED')

15:45:13  INFO      [v7.6] Validation: 7 checks, 7 passed, 0 failed
=== v7.6 Validation Report ===
                                                                check  passed                                                                                                             detail
                                StyleCode totals == StyleColor totals    True                                                                                                    max_diff=0.0009
                                      StyleColor totals == SKU totals    True                                                                                                    max_diff=0.0003
                                            No negative ForecastUnits    True                                                                                                    0 negative rows
                   No duplicate (SKU, MonthStart, HorizonMonths) rows    True                                                    

## Section 10 — Holdout Evaluation (Jan–Feb 2026)

In [14]:
from lane7_forecast.production_outputs_v76 import (
    score_v76_holdout, build_v76_error_decomposition,
)

actuals_df = actuals_df_full

v76_eval, v76_preds = score_v76_holdout(
    sku_fc=prod_sku, actuals_df=actuals_df, holdout_months=HOLDOUT_MONTHS,
)
v76_eval.to_csv( OUTPUT_DIR/'v7_6_holdout_evaluation.csv',   index=False)
v76_preds.to_csv(OUTPUT_DIR/'v7_6_holdout_predictions.csv',  index=False)

v76_decomp = {}
for horizon in [3, 1]:
    if horizon not in v76_cal_fc: continue
    v76_decomp[horizon] = build_v76_error_decomposition(
        actuals_df=actuals_df, dim_product_df=dim_prod_df,
        scode_fc=v76_cal_fc[horizon],
        scol_fc=v76_scol_fc[horizon],
        sku_fc=v76_sku_fc[horizon],
        holdout_months=HOLDOUT_MONTHS,
    )

decomp_all = pd.concat([df for df in v76_decomp.values() if not df.empty], ignore_index=True)
decomp_all.to_csv(OUTPUT_DIR/'v7_6_error_decomposition.csv', index=False)

print('=== v7.6 HOLDOUT EVALUATION ===')
print(v76_eval.to_string(index=False))
print()
print('=== ERROR DECOMPOSITION ===')
print(decomp_all.sort_values(['HorizonMonths','MonthStart','Level']).to_string(index=False))

=== v7.6 HOLDOUT EVALUATION ===
Level  HorizonMonths MonthStart Segment  N_SKUs  TotalActual  TotalPredicted  AbsError   WMAPE
  SKU              1    2026-01     ALL    1976       949368       920523.96 521707.71 54.9532
  SKU              1    2026-02     ALL    2031       833964       993736.99 510798.85 61.2495
  SKU              1    2026-03     ALL    2092       950719      1038071.99 668219.74 70.2857
  SKU              1    2026-04     ALL    2034       808962       990876.04 535176.55 66.1560
  SKU              3    2026-01     ALL    1976       949368       856363.00 498247.79 52.4821
  SKU              3    2026-02     ALL    2031       833964       884205.02 458893.69 55.0256
  SKU              3    2026-03     ALL    2092       950719       944932.88 613097.47 64.4878
  SKU              3    2026-04     ALL    2034       808962       919713.36 504611.70 62.3777

=== ERROR DECOMPOSITION ===
     Level  HorizonMonths MonthStart  TotalActual  TotalPredicted   WMAPE
       SKU

## Section 11 — Bias Analysis

In [15]:
from lane7_forecast.global_bias_control_v76 import build_v76_bias_analysis

if 3 in v76_raw_fc and 3 in v76_cal_fc:
    bias_analysis = build_v76_bias_analysis(
        actuals_df=actuals_df,
        dim_product_df=dim_prod_df,
        raw_scode_fc=v76_raw_fc[3],
        calibrated_scode_fc=v76_cal_fc[3],
        holdout_months=HOLDOUT_MONTHS,
        calibration_df=calib_table,
    )
    bias_analysis.to_csv(OUTPUT_DIR/'v7_6_bias_analysis.csv', index=False)

    print('=== v7.6 BIAS ANALYSIS ===')
    print(bias_analysis.to_string(index=False))
    print()
    print('BiasImprovement > 0 = calibration moved bias closer to 1.0')

=== v7.6 BIAS ANALYSIS ===
 HorizonMonths MonthStart  TotalActual  TotalForecastRaw  TotalForecastCalibrated  RawBiasRatio  CalibratedBiasRatio  BiasImprovement  CalibrationApplied
             3    2026-01     949368.0         971937.05                971937.05        1.0238               1.0238              0.0               False
             3    2026-02     831286.0         997566.49                997566.49        1.2000               1.2000              0.0               False
             3    2026-03     950719.0        1064801.49               1064801.49        1.1200               1.1200              0.0               False
             3    2026-04     808962.0        1052548.39               1052548.39        1.3011               1.3011              0.0               False

BiasImprovement > 0 = calibration moved bias closer to 1.0


## Section 12 — Version Comparison

In [16]:
from lane7_forecast.production_outputs_v76 import build_v76_version_comparison

_prior = next((OUTPUT_DIR/f for f in
               ['v7_5_vs_prior_versions.csv','v7_4_vs_prior_versions.csv',
                'v7_3_vs_v7_2_comparison.csv']
               if (OUTPUT_DIR/f).exists()), None)

_v74_eval = pd.read_csv(OUTPUT_DIR/'v7_4_holdout_evaluation.csv') \
    if (OUTPUT_DIR/'v7_4_holdout_evaluation.csv').exists() else None
_v75_eval = pd.read_csv(OUTPUT_DIR/'v7_5_holdout_evaluation.csv') \
    if (OUTPUT_DIR/'v7_5_holdout_evaluation.csv').exists() else None

comparison = build_v76_version_comparison(
    v76_holdout_eval=v76_eval,
    v76_decomp=decomp_all,
    actuals_df=actuals_df,
    prior_comparison_path=_prior,
    v74_holdout_eval=_v74_eval,
    v75_holdout_eval=_v75_eval,
    calibration_df=calib_table,
)
comparison.to_csv(OUTPUT_DIR/'v7_6_vs_prior_versions.csv', index=False)

print('=== v7.6 vs PRIOR VERSIONS ===')
print(comparison[[
    'Variant','H3_Jan_WMAPE','H3_Feb_WMAPE','Overall_H3_WMAPE',
    'BiasRatio','ProductionCandidateFlag','Notes'
]].to_string(index=False))

=== v7.6 vs PRIOR VERSIONS ===
                     Variant  H3_Jan_WMAPE  H3_Feb_WMAPE  Overall_H3_WMAPE  BiasRatio  ProductionCandidateFlag            Notes
             v7.4_production       52.4821       55.0256               NaN        NaN                     True                 
  v7.5_calibrated_production       55.2198       60.4068           57.8133     1.1584                     True                 
v7.6_conservative_production       52.4821       55.0256           53.7538     1.0653                     True H[1, 3] rejected


## Section 13 — Client Readiness Summary

In [17]:
print("=" * 70)
print("LANE SEVEN DEMAND FORECASTING — v7.6 CLIENT READINESS SUMMARY")
print("=" * 70)
print(
    f"Evaluation: Detected Holdout Months "
    f"({HOLDOUT_MONTHS[0].strftime('%Y-%m')} to {HOLDOUT_MONTHS[-1].strftime('%Y-%m')})"
)
print("Calibration: Global (per HorizonMonths) with no-regression gate")
print("Allocation : v7.2 recency_only")
print(f"Factor bounds: [{V76_CALIB_CONFIG['min_factor']}, {V76_CALIB_CONFIG['max_factor']}]")
print("=" * 70)
print()

def _wm(df, h, m):
    if df is None or df.empty:
        return float("nan")
    r = df[(df["HorizonMonths"] == h) & (df["MonthStart"] == m)]
    return round(float(r["WMAPE"].iloc[0]), 2) if not r.empty else float("nan")

print("Holdout WMAPE by month:")
for _, row in v76_eval.sort_values(["HorizonMonths", "MonthStart"]).iterrows():
    print(
        f'  H={int(row["HorizonMonths"])} '
        f'{row["MonthStart"]}: {row["WMAPE"]:.1f}%'
    )
print()

tot_act = v76_eval["TotalActual"].sum()
tot_pred = v76_eval["TotalPredicted"].sum()
bias_v76 = round(tot_pred / tot_act, 4) if tot_act > 0 else float("nan")

print(f"Overall BiasRatio: {bias_v76:.4f}")
print()

if not calib_table.empty:
    for _, row in calib_table.iterrows():
        h = int(row["HorizonMonths"])
        status = "ACCEPTED" if row["calibration_applied"] else f"REJECTED ({row['rejection_reason']})"
        print(f"  H={h} calibration: {status}")
print()

if _v74_eval is not None:
    print("Comparison vs v7.4:")
    for _, row in v76_eval.sort_values(["HorizonMonths", "MonthStart"]).iterrows():
        h = int(row["HorizonMonths"])
        m = row["MonthStart"]
        v76_w = float(row["WMAPE"])
        v74_w = _wm(_v74_eval, h, m)

        if not np.isnan(v74_w):
            print(
                f"  H={h} {m}: v7.6 {v76_w:.1f}% vs v7.4 {v74_w:.1f}% "
                f"({v76_w - v74_w:+.1f}pp)"
            )
print()

try:
    _v75_eval = pd.read_csv(OUTPUT_DIR / "v7_5_holdout_evaluation.csv")
except Exception:
    _v75_eval = None

if _v75_eval is not None:
    print("Comparison vs v7.5:")
    for _, row in v76_eval.sort_values(["HorizonMonths", "MonthStart"]).iterrows():
        h = int(row["HorizonMonths"])
        m = row["MonthStart"]
        v76_w = float(row["WMAPE"])
        v75_w = _wm(_v75_eval, h, m)

        if not np.isnan(v75_w):
            print(
                f"  H={h} {m}: v7.6 {v76_w:.1f}% vs v7.5 {v75_w:.1f}% "
                f"({v76_w - v75_w:+.1f}pp)"
            )
print()

green, amber, red = [], [], []

primary_month = HOLDOUT_MONTHS[0].strftime("%Y-%m")
primary_h3 = _wm(v76_eval, 3, primary_month)

if not np.isnan(primary_h3):
    if primary_h3 < 55:
        green.append(f"H=3 {primary_month} WMAPE {primary_h3:.1f}% < 55%")
    elif primary_h3 < 70:
        amber.append(f"H=3 {primary_month} WMAPE {primary_h3:.1f}% in 55–70% range")
    else:
        red.append(f"H=3 {primary_month} WMAPE {primary_h3:.1f}% > 70%")

if abs(bias_v76 - 1.0) < 0.05:
    green.append(f"Bias near-neutral (ratio={bias_v76:.3f})")
elif abs(bias_v76 - 1.0) < 0.10:
    amber.append(f"Mild bias (ratio={bias_v76:.3f})")
else:
    red.append(f"Significant bias (ratio={bias_v76:.3f})")

if val_report is not None and (~val_report["passed"]).sum() == 0:
    green.append("All validation checks passed")
else:
    red.append("Validation failure(s)")

for g in green:
    print(f"  ✓ {g}")
for a in amber:
    print(f"  ⚠ {a}")
for r in red:
    print(f"  ✗ {r}")

print()
if not red:
    if not amber:
        print("✅ VERDICT: v7.6 is READY for client presentation.")
    else:
        print("⚠ VERDICT: v7.6 is CONDITIONALLY READY. Review amber signals.")
else:
    print("❌ VERDICT: v7.6 needs further review before client delivery.")

LANE SEVEN DEMAND FORECASTING — v7.6 CLIENT READINESS SUMMARY
Evaluation: Detected Holdout Months (2026-01 to 2026-04)
Calibration: Global (per HorizonMonths) with no-regression gate
Allocation : v7.2 recency_only
Factor bounds: [0.95, 1.05]

Holdout WMAPE by month:
  H=1 2026-01: 55.0%
  H=1 2026-02: 61.2%
  H=1 2026-03: 70.3%
  H=1 2026-04: 66.2%
  H=3 2026-01: 52.5%
  H=3 2026-02: 55.0%
  H=3 2026-03: 64.5%
  H=3 2026-04: 62.4%

Overall BiasRatio: 1.0653

  H=1 calibration: REJECTED (insufficient_safe_evidence)
  H=3 calibration: REJECTED (insufficient_safe_evidence)

Comparison vs v7.4:
  H=1 2026-01: v7.6 55.0% vs v7.4 55.0% (+0.0pp)
  H=1 2026-02: v7.6 61.2% vs v7.4 61.2% (-0.0pp)
  H=1 2026-03: v7.6 70.3% vs v7.4 70.3% (-0.0pp)
  H=1 2026-04: v7.6 66.2% vs v7.4 66.2% (-0.0pp)
  H=3 2026-01: v7.6 52.5% vs v7.4 52.5% (+0.0pp)
  H=3 2026-02: v7.6 55.0% vs v7.4 55.0% (-0.0pp)
  H=3 2026-03: v7.6 64.5% vs v7.4 64.5% (-0.0pp)
  H=3 2026-04: v7.6 62.4% vs v7.4 62.4% (-0.0pp)

Compariso

---
## Run Order

| Cell | Section | Required? |
|------|---------|----------|
| 02 | 0 | Once |
| 03 | 0 | Always |
| **04** | **1 Config** | **Always** |
| **05** | **2 Panel** | **Required** |
| **06** | **2 CV** | **Required** |
| **07** | **3 Evidence** | **Required** |
| **08** | **4 Raw forecasts** | **Required** |
| **09** | **5 Calibration table** | **Required** |
| **10** | **6 Apply calibration** | **Required** |
| **11** | **7 Allocation** | **Required** |
| **12** | **8 Production SKU** | **Required** |
| **13** | **9 Validation** | **Required** |
| **14** | **10 Holdout eval** | **Required** |
| **15** | **11 Bias analysis** | **Required** |
| **16** | **12 Comparison** | **Required** |
| 17 | 13 Summary | Optional |

**Minimum run order:** `03 → 04 → 05 → 06 → 07 → 08 → 09 → 10 → 11 → 12 → 13 → 14 → 15 → 16`